# Evaluate LSTM on Colab

This notebook sets up Colab, copies `data/lstm_processed/lstm_v1` to local disk, and launches the reusable source evaluation script `src/training/lstm_training/evaluate.py`.

## 1. Mount Drive

In [1]:
%cd /content
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Paths

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# Code should live on local Colab disk. Drive is only for large data/checkpoints/logs.
# Set this to your GitHub repo URL before running the cell.
GITHUB_REPO_URL = "https://github.com/sungjelly/Seoul_bike_project.git"
GITHUB_BRANCH = "main"
CODE_DIR = Path("/content/Seoul_bike_project")
RELOAD_LOCAL_CODE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/Seoul_bike_project")
DATASET_NAME = "lstm_v1"

if CODE_DIR.exists() and RELOAD_LOCAL_CODE:
    shutil.rmtree(CODE_DIR)

if not CODE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(CODE_DIR)],
        check=True,
    )

PROJECT_DIR = CODE_DIR
BASE_DATASET_NAME = "base"
DRIVE_PROCESSED_ROOT = DRIVE_ROOT / "data" / "lstm_processed"
LOCAL_PROCESSED_ROOT = Path("/content/lstm_processed")
DRIVE_DATA_DIR = DRIVE_PROCESSED_ROOT / DATASET_NAME
DRIVE_BASE_DATA_DIR = DRIVE_PROCESSED_ROOT / BASE_DATASET_NAME
LOCAL_DATA_DIR = LOCAL_PROCESSED_ROOT / DATASET_NAME
LOCAL_BASE_DATA_DIR = LOCAL_PROCESSED_ROOT / BASE_DATASET_NAME
DATA_DIR = LOCAL_DATA_DIR

CHECKPOINT_PATH = DRIVE_ROOT / "checkpoints" / "lstm_v1" / "best.pt"
LOG_DIR = DRIVE_ROOT / "logs" / "lstm_v1"

os.chdir(PROJECT_DIR)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("Code directory:", PROJECT_DIR)
print("Drive artifact root:", DRIVE_ROOT)
print("Drive dataset:", DRIVE_DATA_DIR)
print("Drive base dataset:", DRIVE_BASE_DATA_DIR)
print("Local dataset:", DATA_DIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Current working directory:", Path.cwd())

## 3. Install Requirements

In [ ]:
%cd /content/Seoul_bike_project
!python -m pip install -r requirements.txt

## 4. Copy Dataset to Colab Disk

In [ ]:
import shutil

RELOAD_LOCAL_DATA = True
for drive_dir, local_dir in [(DRIVE_BASE_DATA_DIR, LOCAL_BASE_DATA_DIR), (DRIVE_DATA_DIR, LOCAL_DATA_DIR)]:
    if not drive_dir.exists():
        raise FileNotFoundError(f"Missing dataset directory: {drive_dir}")
    if local_dir.exists() and RELOAD_LOCAL_DATA:
        shutil.rmtree(local_dir)
    if not local_dir.exists():
        shutil.copytree(drive_dir, local_dir)
print("Using local dataset:", DATA_DIR)

## 5. Evaluate With Source Script

In [ ]:
EVAL_SPLIT = "test_2025_winter"  # val, test_2025_winter, test_2024_april_june

!python -m src.training.lstm_training.evaluate \
  --config configs/lstm_v1.yaml \
  --data_dir /content/lstm_processed/lstm_v1 \
  --checkpoint_path /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_v1/best.pt \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_v1 \
  --split {EVAL_SPLIT} \
  --batch_size 32768 \
  --wandb_enabled true